# Resource usage versus asks: friction V0 runs

Scratch audit of the jobmon runs in the friction node (`friction_dir` in `paths.yaml`):
the full run in `working/` and the probes in `scratch/probe_v0/`.

Measures per task:

- **asked**: memory and wall limit the task ran under, from sacct (`req_mem_gib`, `timelimit_minutes` in the slot's `audit*.csv`, written by `build-friction audit`);
- **used, memory**: the largest per-block process peak RSS of the task (REPORT lines of every attempt, `launch.report_peaks`), not sacct's MaxRSS, which is the cgroup peak and fills with file cache up to the ask;
- **used, runtime**: the task's wall time of its last attempt (sacct `elapsed_seconds`) and its block compute time (sum of `seconds_total` over the blocks it registered in its atomic sidecar, `launch.block_compute_minutes`);
- **group**: the task's memory tier (`mem_tier` in its task features, from the predicted block peak).

The criterion (Bobby, 2026-11-03): the mean used/asked per axis near 0.65, no tier systematically off, a small rerun rate.

In [ ]:
from pathlib import Path

import json
import pandas as pd

from idd_travel_time.friction import launch as ln
from idd_travel_time.paths import load_paths

pd.set_option("display.width", 200)
pd.set_option("display.precision", 2)
node = Path(load_paths()["friction_dir"])
slots = {"working": node / "working", "probes": node / "scratch" / "probe_v0"}
{k: v.exists() for k, v in slots.items()}

In [ ]:
def task_frame(slot: Path) -> pd.DataFrame:
    """One row per task and workflow audited in *slot*, with asks and measured use."""
    audits = ln.slot_audits(slot)
    peaks = ln.report_peaks(slot / "logs")
    compute = ln.block_compute_minutes(slot)
    rows = []
    for r in audits.itertuples(index=False):
        feats = json.loads(r.task_features_json) if isinstance(r.task_features_json, str) else {}
        tid = str(r.task_id)
        rows.append(
            {
                "workflow": int(r.workflow_id),
                "task_id": tid,
                "tier": feats.get("mem_tier", "?"),
                "n_blocks": feats.get("n_blocks"),
                "state": r.state,
                "asked_gib": float(r.req_mem_gib),
                "peak_gib": float(peaks.get(tid, float("nan"))),
                "sacct_gib": float(r.max_rss_gib),
                "asked_min": float(r.timelimit_minutes),
                "wall_min": float(r.elapsed_seconds) / 60.0,
                "compute_min": float(compute.get(tid, float("nan"))),
            }
        )
    df = pd.DataFrame(rows)
    df["mem_used_over_asked"] = df["peak_gib"] / df["asked_gib"]
    df["time_used_over_asked"] = df["wall_min"] / df["asked_min"]
    return df


frames = {name: task_frame(slot) for name, slot in slots.items()}
tasks = pd.concat(frames.values(), ignore_index=True)
tasks.groupby("workflow").size().rename("tasks_audited")

In [ ]:
def attempts(slot: Path) -> pd.DataFrame:
    """Every Slurm attempt saved beside the audits (`audit_attempts_<workflow>.csv`, sacct --parsable2), by state.

    The header line is located by name: some of these files were sorted after sacct wrote them.
    """
    out = []
    for path in sorted(slot.glob("audit_attempts_*.csv")):
        wf = int(path.stem.rsplit("_", 1)[1])
        lines = [ln_ for ln_ in path.read_text().splitlines() if ln_.strip()]
        header = next(ln_ for ln_ in lines if ln_.startswith("JobID"))
        rows = [ln_.split("|") for ln_ in lines if ln_ != header]
        a = pd.DataFrame(rows, columns=header.split("|"))
        a = a[a["JobName"].str.startswith("friction_block")]
        counts = a["State"].value_counts()
        out.append({"workflow": wf, "attempts": int(len(a)), **{s: int(n) for s, n in counts.items()}})
    return pd.DataFrame(out).fillna(0).astype({"attempts": int}).set_index("workflow")


pd.concat([attempts(s) for s in slots.values()]).fillna(0).astype(int).sort_index()

## Overall, per workflow

`asked_hours` and `used_hours` are task-hours (sum of limits, sum of wall); `asked_gib_hours` and `peak_gib_hours` weight memory by time.

In [ ]:
def overall(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("workflow")
    return pd.DataFrame(
        {
            "tasks": g.size(),
            "mem_used_over_asked_mean": g["mem_used_over_asked"].mean(),
            "mem_min": g["mem_used_over_asked"].min(),
            "mem_max": g["mem_used_over_asked"].max(),
            "time_used_over_asked_mean": g["time_used_over_asked"].mean(),
            "time_min": g["time_used_over_asked"].min(),
            "time_max": g["time_used_over_asked"].max(),
            "asked_hours": g["asked_min"].sum() / 60,
            "used_hours": g["wall_min"].sum() / 60,
            "asked_gib_hours": g.apply(lambda d: (d["asked_gib"] * d["asked_min"]).sum() / 60),
            "peak_gib_hours": g.apply(lambda d: (d["peak_gib"] * d["wall_min"]).sum() / 60),
        }
    )


overall(tasks)

## By memory tier, per workflow

In [ ]:
by_tier = tasks.groupby(["workflow", "tier"]).agg(
    tasks=("task_id", "size"),
    asked_gib=("asked_gib", "mean"),
    peak_gib=("peak_gib", "mean"),
    mem_used_over_asked=("mem_used_over_asked", "mean"),
    asked_min=("asked_min", "mean"),
    wall_min=("wall_min", "mean"),
    compute_min=("compute_min", "mean"),
    time_used_over_asked=("time_used_over_asked", "mean"),
)
by_tier

## idd-tools' own summaries on the sacct-joined audit

`policy_metrics` and `sizing_floor_report` read the audit frame as `audit_run` wrote it (sacct zones: memory by MaxRSS, which fills to the ask here, so its memory zone reads "under"); `cost_by_feature` groups a cost frame by a feature, here fed the process peak and wall time per task.

In [ ]:
from idd_tools.jobmon import cost_by_feature
from idd_tools.jobmon.resource_audit import policy_metrics, sizing_floor_report

for wf in sorted(tasks["workflow"].unique()):
    slot = slots["working"] if wf == 648580 else slots["probes"]
    audit = ln.slot_audits(slot, workflows=[wf])
    print(f"workflow {wf}: sacct zones {policy_metrics(audit)['overall'].round(2).to_dict()}; "
          f"sizing floor {sizing_floor_report(audit)}")
cost = tasks.rename(columns={"peak_gib": "jobmon_mem_gib"}).assign(jobmon_runtime_s=tasks["wall_min"] * 60)
cost_by_feature(cost[cost["workflow"] == 648580], by="tier")

## Reading the tables

- Memory: judge `mem_used_over_asked` (process peak over ask). The sacct column `sacct_gib` sits at the ask on most tasks and says nothing about need.
- Runtime: `time_used_over_asked` is wall over limit for the last attempt; a task that timed out and resumed shows a short last attempt, so read it with the attempts table. `compute_min` is the uncensored block build time.
- A tier whose mean sits far from the others is a systematic miss of the model for that group.